# 🚀 04 — OTA Deploy Dashboard

Interactive firmware deployment panel.  
Point it at your build output folder, run the two setup cells, then **push one button** to flash any bot.

```
firmware/
  build/
    dogbot_v1.bin      ← matched to bots with platform="dogbot_v1"
    rfbot.bin          ← matched to rfbot* hostnames
    simplebot.bin
    ...
```

> **Requires:** `pip install ipywidgets`  (already included if you used `pip install fleet-manager[notebooks]`)

---

## ① Configure paths  *(edit these)*

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
FLEET_MGR = REPO_ROOT / 'fleet-manager'
if str(FLEET_MGR) not in sys.path:
    sys.path.insert(0, str(FLEET_MGR))

# ── Set your paths here ───────────────────────────────────────────────────────

# Folder containing your compiled .bin files
# e.g. REPO_ROOT / 'firmware' / 'build'
FIRMWARE_DIR = REPO_ROOT / 'firmware' / 'build'

# fleet.yaml for shipped-bot filtering
FLEET_YAML   = REPO_ROOT / 'docs' / 'fleet.yaml'

# Which bot families to scan
BOT_FAMILIES = ['rfbot', 'mybot', 'carbot', 'paulbot', 'simplebot', 'dogbot']

# Scan concurrency
SCAN_WORKERS = 20
SCAN_TIMEOUT = 1.0

print(f'Firmware dir : {FIRMWARE_DIR}  (exists={FIRMWARE_DIR.exists()})')
print(f'fleet.yaml   : {FLEET_YAML}    (exists={FLEET_YAML.exists()})')

---
## ② Scan fleet + firmware  *(run once, or re-run after a build)*

In [ ]:
import time
from fleet_manager import scan, probe_fleet, scan_firmware_dir, match_firmware_to_fleet

# ── Scan online bots ──────────────────────────────────────────────────────────
print('Scanning fleet…', end=' ', flush=True)
t0 = time.perf_counter()
fleet = scan(BOT_FAMILIES, workers=SCAN_WORKERS, timeout=SCAN_TIMEOUT, yaml_path=FLEET_YAML)
print(f'done in {time.perf_counter()-t0:.2f}s  —  {fleet.summary()}')

# ── Probe firmware versions from running bots ─────────────────────────────────
if fleet.online:
    print(f'Probing {len(fleet.online)} online bot(s) for current firmware…', end=' ', flush=True)
    probe_fleet(fleet, timeout=2.0)
    print('done.')

# ── Scan build folder ─────────────────────────────────────────────────────────
fw_map: dict = {}
if FIRMWARE_DIR.exists():
    fw_map = scan_firmware_dir(FIRMWARE_DIR)
    print(f'\nFound {len(fw_map)} firmware file(s) in {FIRMWARE_DIR}:')
    for name, path in fw_map.items():
        size_kb = path.stat().st_size / 1024
        print(f'  {name:<20}  {path.name:<30}  {size_kb:>7.1f} KB')
else:
    print(f'\n⚠️  Firmware directory not found: {FIRMWARE_DIR}')
    print('    Continuing without firmware auto-match — you can still deploy manually.')

# ── Auto-match firmware to bots ───────────────────────────────────────────────
pairs = match_firmware_to_fleet(fleet, fw_map) if fw_map else {}

print(f'\nAuto-matched {len(pairs)}/{len(fleet.online)} online bot(s) to a firmware binary.')

---
## ③ Deploy Dashboard  *(push a button to flash)*

In [ ]:
try:
    import ipywidgets as w
    from IPython.display import display, HTML
except ImportError:
    print('ipywidgets not installed. Run: pip install ipywidgets')
    raise

import threading
from fleet_manager import ota_update

# ── Styles ────────────────────────────────────────────────────────────────────
CSS = """
<style>
.ota-card {
    background: #1e293b;
    border: 1px solid #334155;
    border-radius: 10px;
    padding: 14px 18px;
    margin: 6px 0;
    font-family: 'JetBrains Mono', 'Fira Code', monospace;
    display: flex;
    align-items: center;
    gap: 12px;
}
.ota-hostname  { color: #e2e8f0; font-weight: bold; min-width: 180px; }
.ota-platform  { color: #94a3b8; font-size: 0.85em; min-width: 110px; }
.ota-fw-old    { color: #64748b; font-size: 0.82em; min-width: 80px; }
.ota-fw-new    { color: #60a5fa; font-size: 0.82em; min-width: 160px; }
.ota-ip        { color: #475569; font-size: 0.78em; min-width: 120px; }
.ota-status    { font-size: 0.85em; min-width: 90px; text-align: right; }
.ota-ok        { color: #4ade80; }
.ota-err       { color: #f87171; }
.ota-busy      { color: #facc15; }
.ota-pending   { color: #475569; }
.ota-header {
    background: #0f172a;
    border-radius: 10px;
    padding: 10px 18px;
    margin-bottom: 4px;
    color: #60a5fa;
    font-family: monospace;
    font-size: 0.82em;
    display: flex;
    gap: 12px;
}
.ota-log-box {
    background: #0f172a;
    border: 1px solid #1e293b;
    border-radius: 8px;
    padding: 10px 14px;
    font-family: monospace;
    font-size: 0.82em;
    color: #94a3b8;
    max-height: 220px;
    overflow-y: auto;
}
</style>
"""
display(HTML(CSS))

# ── Per-bot state ─────────────────────────────────────────────────────────────
DEPLOY_STATE: dict = {}   # hostname → 'idle' | 'busy' | 'ok' | 'error'

# ── Shared log ────────────────────────────────────────────────────────────────
log_out = w.Output()
log_lines: list[str] = []

def _log(msg: str) -> None:
    log_lines.append(msg)
    with log_out:
        log_out.clear_output(wait=True)
        display(HTML(
            '<div class="ota-log-box">' +
            '<br>'.join(log_lines[-50:]) +   # keep last 50 lines
            '</div>'
        ))


# ── Build one card per bot ────────────────────────────────────────────────────
all_bots = fleet.online

# Also include unmatched online bots (user can pick firmware manually)
unmatched = [b for b in all_bots if b.hostname not in pairs]

def _make_deploy_button(label: str, style: str = 'primary') -> w.Button:
    btn = w.Button(
        description=label,
        button_style=style,
        layout=w.Layout(width='120px', height='32px'),
    )
    return btn


def _make_bot_card(bot, fw_path: Path | None, fw_picker: w.Dropdown | None):
    hostname = bot.hostname
    DEPLOY_STATE[hostname] = 'idle'

    # Status HTML label
    status_html = w.HTML(value='<span class="ota-status ota-pending">—</span>')

    def _set_status(state: str):
        icons = {'idle': '—', 'busy': '⟳ Flashing…', 'ok': '✓ Done', 'error': '✗ Failed'}
        classes = {'idle': 'ota-pending', 'busy': 'ota-busy', 'ok': 'ota-ok', 'error': 'ota-err'}
        DEPLOY_STATE[hostname] = state
        status_html.value = (
            f'<span class="ota-status {classes[state]}">{icons[state]}</span>'
        )

    # Deploy button
    deploy_btn = _make_deploy_button('⚡ Deploy')

    def _on_deploy(b):
        # Resolve firmware path
        if fw_path is not None:
            target_fw = fw_path
        elif fw_picker is not None and fw_picker.value:
            target_fw = Path(fw_picker.value)
        else:
            _log(f'[{hostname}] ✗  No firmware selected.')
            _set_status('error')
            return

        if not target_fw.exists():
            _log(f'[{hostname}] ✗  File not found: {target_fw}')
            _set_status('error')
            return

        deploy_btn.disabled = True
        _set_status('busy')
        size_kb = target_fw.stat().st_size / 1024
        _log(f'[{hostname}]  → Uploading {target_fw.name}  ({size_kb:.1f} KB)…')

        def _do_upload():
            ok = ota_update(bot, target_fw, timeout=90.0)
            _set_status('ok' if ok else 'error')
            result = '✓ Success' if ok else '✗ Failed'
            _log(f'[{hostname}]  {result}  ({target_fw.name})')
            deploy_btn.disabled = False

        threading.Thread(target=_do_upload, daemon=True).start()

    deploy_btn.on_click(_on_deploy)

    # Card content
    fw_label = fw_path.name if fw_path else '(no match)'
    fw_color = '#60a5fa' if fw_path else '#ef4444'
    cur_fw   = bot.fw_version or '—'
    plat     = bot.platform   or '—'

    info_html = w.HTML(value=(
        f'<div style="display:flex; gap:12px; align-items:center;">'
        f'<span class="ota-hostname">{hostname}</span>'
        f'<span class="ota-ip">{bot.ip}</span>'
        f'<span class="ota-platform">{plat}</span>'
        f'<span class="ota-fw-old">v{cur_fw}</span>'
        f'<span class="ota-fw-new" style="color:{fw_color};">→ {fw_label}</span>'
        f'</div>'
    ))

    row_children = [info_html]
    if fw_picker is not None:
        row_children.append(fw_picker)
    row_children += [deploy_btn, status_html]

    card = w.HBox(
        row_children,
        layout=w.Layout(
            padding='10px 16px',
            margin='4px 0',
            border='1px solid #334155',
            border_radius='8px',
            align_items='center',
        ),
    )
    return card, deploy_btn, _set_status


# ── Assemble dashboard ────────────────────────────────────────────────────────
all_cards    = []
all_btns     = []
all_statuses = []

header_html = w.HTML(value=(
    '<div class="ota-header">'
    '<span style="min-width:180px;">Hostname</span>'
    '<span style="min-width:120px;">IP</span>'
    '<span style="min-width:110px;">Platform</span>'
    '<span style="min-width:80px;">Current</span>'
    '<span style="min-width:160px;">Target firmware</span>'
    '</div>'
))
all_cards.append(header_html)

# Matched bots (auto-selected firmware)
for hostname, (bot, fw_path) in pairs.items():
    card, btn, set_st = _make_bot_card(bot, fw_path, None)
    all_cards.append(card)
    all_btns.append(btn)
    all_statuses.append(set_st)

# Unmatched bots (manual firmware picker)
fw_options = [('(select firmware)', '')] + [
    (f'{name}  ({p.name})', str(p)) for name, p in fw_map.items()
]
for bot in unmatched:
    picker = w.Dropdown(
        options=fw_options,
        value='',
        layout=w.Layout(width='220px'),
    )
    card, btn, set_st = _make_bot_card(bot, None, picker)
    all_cards.append(card)
    all_btns.append(btn)
    all_statuses.append(set_st)

if not all_bots:
    all_cards.append(w.HTML(
        '<div style="color:#ef4444;padding:16px;">'
        '⚠️  No online bots found. Re-run cell ② after powering them on.'
        '</div>'
    ))

# ── Deploy All button ─────────────────────────────────────────────────────────
deploy_all_btn = w.Button(
    description='⚡ Deploy ALL',
    button_style='danger',
    layout=w.Layout(width='150px', height='36px'),
    tooltip='Flash every bot simultaneously',
)
rescan_btn = w.Button(
    description='🔄 Re-scan',
    button_style='',
    layout=w.Layout(width='110px', height='36px'),
    tooltip='Re-run cell ② to refresh fleet + firmware',
)
clear_log_btn = w.Button(
    description='🗑 Clear log',
    button_style='',
    layout=w.Layout(width='110px', height='36px'),
)
fleet_label = w.HTML(
    f'<span style="color:#94a3b8;font-family:monospace;font-size:0.88em;">'
    f'{len(all_bots)} online  ·  {len(pairs)} auto-matched  ·  {len(fw_map)} firmware files'
    f'</span>'
)

def _on_deploy_all(b):
    for btn in all_btns:
        if not btn.disabled:
            btn.click()

def _on_clear_log(b):
    log_lines.clear()
    with log_out:
        log_out.clear_output()

def _on_rescan(b):
    _log('ℹ️  Re-run cell ② (Scan fleet + firmware) to refresh the fleet.')

deploy_all_btn.on_click(_on_deploy_all)
clear_log_btn.on_click(_on_clear_log)
rescan_btn.on_click(_on_rescan)

toolbar = w.HBox(
    [deploy_all_btn, rescan_btn, clear_log_btn, fleet_label],
    layout=w.Layout(gap='8px', align_items='center', margin='8px 0'),
)

# ── Title ─────────────────────────────────────────────────────────────────────
title_html = w.HTML(
    '<div style="'
    'background:linear-gradient(135deg,#1e3a5f,#0f172a);'
    'border:1px solid #2563eb;border-radius:10px;'
    'padding:14px 20px;margin-bottom:10px;'
    '">'
    '<span style="font-size:1.3em;font-weight:bold;color:#e2e8f0;">'
    '🚀 OTA Deploy Dashboard'
    '</span>'
    '<span style="color:#64748b;font-size:0.82em;margin-left:16px;font-family:monospace;">'
    'fleet-manager v2.0'
    '</span>'
    '</div>'
)

log_label = w.HTML('<div style="color:#60a5fa;margin-top:12px;margin-bottom:4px;font-family:monospace;">📋 Deploy log</div>')

dashboard = w.VBox(
    [title_html, toolbar] + all_cards + [log_label, log_out],
    layout=w.Layout(padding='4px'),
)

_log(f'Dashboard ready — {len(all_bots)} online bot(s), {len(fw_map)} firmware file(s).')
display(dashboard)

---
## ④ Manual deploy (no widgets — fallback)

If `ipywidgets` is unavailable or you prefer scripted output:

In [ ]:
# ── Pick a specific bot by hostname, or set to None to deploy all matched ──────
TARGET_HOSTNAME = None   # e.g. 'paulbot0.local'  or None for all matched

from fleet_manager import ota_update

targets = (
    {k: v for k, v in pairs.items() if k == TARGET_HOSTNAME}
    if TARGET_HOSTNAME
    else pairs
)

if not targets:
    print('No targets to deploy. Check TARGET_HOSTNAME or re-run cell ②.')
else:
    print(f'Deploying to {len(targets)} bot(s)…\n')
    for hostname, (bot, fw_path) in targets.items():
        size_kb = fw_path.stat().st_size / 1024
        print(f'  → {hostname:<22} {fw_path.name:<28} ({size_kb:.1f} KB)  ', end='', flush=True)
        t0 = time.perf_counter()
        ok = ota_update(bot, fw_path, timeout=90.0)
        elapsed = time.perf_counter() - t0
        print(f'  {"✓ OK" if ok else "✗ FAILED"}  ({elapsed:.1f}s)')

    success = sum(1 for h, (b, f) in targets.items() if True)  # logged per bot above
    print('\nDone.')

---
## ⑤ Post-deploy verification

In [ ]:
from fleet_manager import scan, probe_fleet

REBOOT_WAIT_S = 15

print(f'Waiting {REBOOT_WAIT_S}s for bots to reboot…')
for remaining in range(REBOOT_WAIT_S, 0, -1):
    print(f'\r  {remaining}s…  ', end='', flush=True)
    time.sleep(1)
print('\r  Done.         ')

# Re-scan the same families
new_fleet = scan(BOT_FAMILIES, workers=SCAN_WORKERS, timeout=SCAN_TIMEOUT, yaml_path=FLEET_YAML)
probe_fleet(new_fleet, timeout=3.0)

# Compare old vs new firmware
old_fw = {b.hostname: b.fw_version for b in fleet.online}

print(f'\n{"Hostname":<24} {"Status":<10} {"Was":<12} {"Now":<12} {""}')
print('─' * 66)
for bot in new_fleet.bots:
    if bot.hostname in old_fw or bot.hostname in pairs:
        was  = old_fw.get(bot.hostname, '—')
        now  = bot.fw_version or '(unknown)'
        flag = '✓ updated' if was and now != was else ('● online' if bot.is_online else '○ offline')
        print(f'  {bot.hostname:<22} {bot.status:<10} {was:<12} {now:<12}  {flag}')